# Video Game Sales Analysis (1980-2020)

Exploratory data analysis of the `vgsales.csv` dataset. It's 16,598 video games with regional sales figures (North America, Europe, Japan, Other), originally scraped from VGChartz. I used the RAWG Video Games API to fill in some missing release years.

This notebook covers data cleaning, exploratory analysis, and a small predictive modeling example looking at whether a game's sales can be predicted from its platform, genre, publisher, and release year.

**Sections:**
1. Data Loading & Cleaning
2. Genre & Platform Trends
3. Publisher Analysis
4. Regional Sales Patterns
5. Platform Lifecycles & Console Manufacturers
6. Can We Predict a Hit? (Modeling)
7. A Quick Time-Series Forecast
8. Clustering Publishers
9. Does Critic Score Help Predict Sales?
10. Discussion
11. Limitations
12. Export for Tableau


## Questions I wanted to answer

- Which genres and platforms have sold the most over the life of this dataset, and has that changed over time?
- Are some publishers more consistent "hit-makers" than others, or is it mostly just about releasing more games?
- Does regional demand differ by genre, or is there no connection?
- Can a game's platform, genre, publisher, and release year actually predict how well it sells?
- How have Nintendo, Sony, and Microsoft traded off over the years?


## 1. Data Loading & Cleaning

### Setup

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import os
import json

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Consistent styling for every chart in this notebook
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

In [ ]:
original_df = pd.read_csv('/kaggle/input/videogamesales/vgsales.csv')
df = original_df.copy()

print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
df.head()

### Checking for missing data

Before doing anything else I wanted to see what was actually missing, instead of just dropping rows or filling blindly.

In [ ]:
null_summary = df.isnull().sum().sort_values(ascending=False)
null_pct = (null_summary / len(df) * 100).round(2)
pd.DataFrame({'missing_count': null_summary, 'missing_pct': null_pct}).query('missing_count > 0')


So:
- `Year` is missing for 271 titles (1.63%), small enough that I didn't want to just drop them.
- `Publisher` is missing for 58 titles (0.35%), which is negligible. I drop these later when publisher matters.
- Nothing else is missing, and the sales columns all look reasonable (no negative values, checked below).

### Cleaning

In [ ]:
# Year as a nullable integer so we can hold pd.NA alongside real years
df['Year'] = df['Year'].astype('Int64')

# Value check: Global_Sales should equal (approximately) the sum of the four regions.
# Small floating-point rounding differences are expected; large gaps would flag a data issue.
implied_global = df[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']].sum(axis=1)
max_gap = (df['Global_Sales'] - implied_global).abs().max()
print(f"Largest gap between Global_Sales and NA+EU+JP+Other: {max_gap:.4f}M units")
print("-> Global_Sales is a derived total, not an independent measurement.")
print("   Meaning NA/EU/JP sales can't be used as predictors of Global_Sales")
print("   without leaking the answer into the model.")


### Filling in missing years with the RAWG API

Since 271 titles don't have a Year, I looked up each one using the RAWG Video Games API instead of just dropping them.

(Small note: I'm pulling from Colab's secrets manager instead. I added a secret called `RAWG_API_KEY`. If there's no key set, this step just gets skipped and those 271 rows gets dropped instead. Everything else still runs fine.)

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("RAWG_API_KEY")
except Exception:
    api_key = None

if not api_key:
    print("No RAWG_API_KEY secret found, skipping year enrichment. "
          "The 271 rows with missing Year will be excluded from year-based analysis below.")

# Simple on-disk cache so re-running the notebook doesn't re-hit the API for titles
# we've already looked up (RAWG's free tier is rate-limited).
cache_path = '/kaggle/working/rawg_year_cache.json'
try:
    with open(cache_path) as f:
        year_cache = json.load(f)
except Exception:
    year_cache = {}

def fetch_year(game_name, api_key):
    if game_name in year_cache:
        return year_cache[game_name]

    url = "https://api.rawg.io/api/games"
    params = {'key': api_key, 'search': game_name}
    try:
        time.sleep(0.1)  # be polite to the free-tier rate limit
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        year = None
        if data.get('results'):
            released = data['results'][0].get('released')
            if released:
                year = released.split('-')[0]
    except Exception as e:
        print(f"Error fetching '{game_name}': {e}")
        year = None

    year_cache[game_name] = year
    return year

if api_key:
    null_mask = df['Year'].isnull()
    print(f"Looking up {null_mask.sum()} missing release years...")
    df.loc[null_mask, 'Year'] = df.loc[null_mask, 'Name'].apply(lambda n: fetch_year(n, api_key))

    with open(cache_path, 'w') as f:
        json.dump(year_cache, f)

    still_missing = df['Year'].isnull().sum()
    print(f"Done. {still_missing} titles still have no year after lookup (not found in RAWG).")

In [ ]:
df['Rank'] = range(1, len(df) + 1)  # sequential rank by row order (ties with original file order)

clean_df = df.dropna(subset=['Year']).copy()
clean_df['Year'] = clean_df['Year'].astype(int)

print(f"Analysis dataset: {len(clean_df):,} rows "
      f"({len(df) - len(clean_df)} rows dropped for unresolvable Year)")
clean_df.describe(include='all').T[['count', 'unique', 'top', 'freq']].fillna('')


## 2. Genre & Platform Trends

Which genres and platforms have sold the most overall, and how has regional demand shifted over time?

In [ ]:
# A fixed, reused color per genre so the same genre is always the same color
# across every chart in this notebook (bar, stacked area, regional split).
genres_by_sales = clean_df.groupby('Genre')['Global_Sales'].sum().sort_values(ascending=False)
genre_palette = dict(zip(genres_by_sales.index, sns.color_palette('tab20', n_colors=genres_by_sales.shape[0])))

top_platforms = clean_df.groupby('Platform')['Global_Sales'].sum().sort_values(ascending=False).head(10)
regional_trends = clean_df.groupby('Year')[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']].sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

genres_by_sales.plot(kind='bar', ax=axes[0], color=[genre_palette[g] for g in genres_by_sales.index])
axes[0].set_title('Global Sales by Genre (All-Time)', fontweight='bold')
axes[0].set_ylabel('Global Sales (Millions)')
axes[0].set_xlabel('')

top_platforms.plot(kind='bar', ax=axes[1], color=sns.color_palette('Blues_r', n_colors=10))
axes[1].set_title('Top 10 Platforms by Global Sales (All-Time)', fontweight='bold')
axes[1].set_ylabel('Global Sales (Millions)')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
regional_trends.plot(marker='o', ax=plt.gca())
plt.title('Regional Sales Over Time', fontweight='bold')
plt.ylabel('Sales (Millions)')
plt.xlabel('Year')
plt.legend(title='Region', labels=['North America', 'Europe', 'Japan', 'Other'])
plt.tight_layout()
plt.show()


Action (1,723M units) and Sports (1,309M) are by far the two biggest genres, followed by Shooter, Role-Playing, and Platform. PS2 is the best-selling platform overall (1,233M), probably a mix of it having a huge install base and an unusually long generation. Sales across every region peak around 2007-2009 and drop off pretty sharply after that.

## 3. Publisher Analysis

Just looking at total sales rewards publishers for releasing more games. Dividing by number of titles gives a better sense of who's actually putting out hits versus who's just producing more.

In [ ]:
publisher_stats = clean_df.groupby('Publisher').agg(
    Total_Sales=('Global_Sales', 'sum'),
    Game_Count=('Name', 'count')
)
publisher_stats['Sales_Per_Game'] = publisher_stats['Total_Sales'] / publisher_stats['Game_Count']

top10_by_total = publisher_stats.sort_values('Total_Sales', ascending=False).head(10)
display(top10_by_total.round(2))

# Hit-makers: highest sales-per-game among publishers with a meaningful catalog (>=10 titles)
# so a single lucky release doesn't top the list.
hitmakers = publisher_stats[publisher_stats['Game_Count'] >= 10].sort_values('Sales_Per_Game', ascending=False).head(15)

plt.figure(figsize=(11, 7))
sizes = (hitmakers['Total_Sales'] / hitmakers['Total_Sales'].max() * 800) + 40
scatter = plt.scatter(hitmakers['Game_Count'], hitmakers['Sales_Per_Game'],
                       s=sizes, c=hitmakers['Sales_Per_Game'], cmap='YlOrRd',
                       edgecolor='white', linewidth=0.8, alpha=0.9)
for name, row in hitmakers.iterrows():
    plt.annotate(name, (row['Game_Count'], row['Sales_Per_Game']),
                 xytext=(6, 4), textcoords='offset points', fontsize=8)
plt.title('Hit-Makers vs. Volume Dealers\n(Publishers with ≥10 titles, bubble size = total sales)', fontweight='bold')
plt.xlabel('Number of Titles Released')
plt.ylabel('Average Global Sales per Title (Millions)')
plt.tight_layout()
plt.show()


Nintendo stands out here: 696 titles averaging 2.56M units each, more than double the next publisher. Makes sense given how much of their catalog is first-party franchises (Mario, Pokémon, Zelda, etc.) and that they release relatively fewer games. EA and Activision are basically the opposite: huge catalogs (1,339 and 966 titles) but much lower averages (0.82M and 0.75M), releasing sports/shooter titles pretty much every year.

## 4. Regional Sales Patterns

Does demand differ by genre across regions, and if so by how much?

In [ ]:
regional_sums = clean_df[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']].sum()
regional_sums.index = ['North America', 'Europe', 'Japan', 'Other']
regional_pct = (regional_sums / regional_sums.sum() * 100).sort_values(ascending=True)

plt.figure(figsize=(9, 4))
bars = plt.barh(regional_pct.index, regional_pct.values, color=sns.color_palette('Blues', n_colors=4))
for bar, pct in zip(bars, regional_pct.values):
    plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2, f'{pct:.1f}%', va='center')
plt.title('Market Share by Region (All-Time)', fontweight='bold')
plt.xlabel('% of Global Sales')
plt.xlim(0, max(regional_pct.values) * 1.15)
plt.tight_layout()
plt.show()


In [ ]:
# Genre preference by region, normalized to 100% per genre
regions = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']
region_labels = {'NA_Sales': 'North America', 'EU_Sales': 'Europe', 'JP_Sales': 'Japan', 'Other_Sales': 'Other'}

regional_genre = clean_df.groupby('Genre')[regions].sum().rename(columns=region_labels)
regional_genre = regional_genre.loc[genres_by_sales.index]  # keep consistent genre ordering
regional_perc = regional_genre.div(regional_genre.sum(axis=1), axis=0) * 100

ax = regional_perc.plot(kind='barh', stacked=True, figsize=(11, 7),
                         color=sns.color_palette('Blues', n_colors=4), edgecolor='white')
plt.title('Regional Sales Split by Genre', fontweight='bold')
plt.xlabel('Share of Global Sales (%)')
plt.ylabel('')
plt.legend(title='Region', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# Genre popularity by year within each region, since 2010 (recent, higher-resolution view)
recent_df = clean_df[clean_df['Year'] >= 2010].copy()

heatmap_cfg = [
    ('NA_Sales', 'North America: Genre Popularity', 'YlOrRd'),
    ('EU_Sales', 'Europe: Genre Popularity', 'BuPu'),
    ('JP_Sales', 'Japan: Genre Popularity', 'YlGnBu'),
]

fig, axes = plt.subplots(1, 3, figsize=(22, 7), sharey=True)
for (col, title, cmap), ax in zip(heatmap_cfg, axes):
    pivot = recent_df.pivot_table(index='Genre', columns='Year', values=col, aggfunc='mean')
    sns.heatmap(pivot, annot=True, fmt=".2f", annot_kws={"size": 8}, cmap=cmap, ax=ax,
                cbar_kws={'label': 'Avg Sales per Title (M)'})
    ax.set_title(title, fontsize=13, pad=14)
    plt.setp(ax.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()


North America makes up 49.2% of global sales, Europe 27.3%, Japan 14.6%, and everywhere else 8.9%. But that overall split hides some genre-level differences. Japan's numbers skew noticeably more toward Role-Playing games than NA/EU do, while Shooter games clearly do well in North America.

## 5. Platform Lifecycles & Console Manufacturers

How has genre popularity shifted from generation to generation, and how have Nintendo, Sony, and Microsoft progressed?

In [ ]:
genre_trends = clean_df.groupby(['Year', 'Genre'])['Global_Sales'].sum().unstack().fillna(0)
genre_trends = genre_trends[genres_by_sales.index]  # consistent genre order/colors

plt.figure(figsize=(13, 6))
plt.stackplot(genre_trends.index, genre_trends.T,
              labels=genre_trends.columns,
              colors=[genre_palette[g] for g in genre_trends.columns], alpha=0.9)
plt.title('Genre Dominance Over Time', fontsize=15, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Global Sales (Millions)')
plt.legend(loc='upper left', title='Genre', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()


In [ ]:
target_platforms = ['PS2', 'DS', 'Wii', 'PS4', 'Switch', 'PS5']
platform_data = clean_df[clean_df['Platform'].isin(target_platforms)]
lifecycle = platform_data.groupby(['Year', 'Platform'])['Global_Sales'].sum().reset_index()

plt.figure(figsize=(13, 6))
sns.lineplot(data=lifecycle, x='Year', y='Global_Sales', hue='Platform', linewidth=3)
plt.title('Platform Lifecycles: Rise and Fall of Iconic Consoles', fontsize=15, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Annual Global Sales (Millions)')
plt.axvspan(2000, 2005, color='gray', alpha=0.1)
plt.axvspan(2017, clean_df['Year'].max(), color='steelblue', alpha=0.06)
plt.legend(title='Platform', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
manufacturer_palette = {'PlayStation': '#2E5EAA', 'Xbox': '#3A9D23', 'Nintendo': '#E4000F'}

def categorize_console(platform):
    if platform in ['PS', 'PS2', 'PS3', 'PS4', 'PS5']:
        return 'PlayStation'
    if platform in ['XB', 'X360', 'XOne']:
        return 'Xbox'
    if platform in ['Wii', 'WiiU', 'NS', 'NES', 'SNES', 'GB', 'GBA', 'DS', '3DS']:
        return 'Nintendo'
    return 'Other'

clean_df['Manufacturer'] = clean_df['Platform'].apply(categorize_console)
mfr_comp = (clean_df[clean_df['Manufacturer'] != 'Other']
            .groupby(['Year', 'Manufacturer'])['Global_Sales'].sum().unstack())

plt.figure(figsize=(13, 6))
for mfr in ['Nintendo', 'PlayStation', 'Xbox']:
    if mfr in mfr_comp.columns:
        plt.plot(mfr_comp.index, mfr_comp[mfr], label=mfr, color=manufacturer_palette[mfr], linewidth=3)
plt.title('Manufacturer Sales Comparison Over Time', fontsize=15, fontweight='bold')
plt.ylabel('Global Sales (Millions)')
plt.xlabel('Year')
plt.legend(title='Manufacturer')
plt.tight_layout()
plt.show()

print("All-time totals:")
print(clean_df[clean_df['Manufacturer'] != 'Other'].groupby('Manufacturer')['Global_Sales'].sum()
      .sort_values(ascending=False).round(1))


Over the whole dataset, PlayStation edges out Nintendo in total sales (3,188M vs. 3,076M), with Xbox well behind at 1,363M. What's more interesting than the totals though is the shape. Nintendo's line has sharp peaks and valleys (Wii, DS) that look very hit-driven, while PlayStation's sales are more evenly sustained across generations. Also worth noting: Action has been the single biggest genre most years since the mid-2000s, and its seems to be growing rather than leveling off.

## 6. Can We Predict a Hit?


My first instinct was to throw every numeric column into a Random Forest to predict Global_Sales. But NA_Sales + EU_Sales + JP_Sales + Other_Sales basically is Global_Sales (see the value check above), so if you use those as features, the model isn't really predicting anything. It's just learning to add. I actually tried this and got R² = 0.82, with 85% of the "predictive power" coming from NA_Sales alone, which is a pretty clear sign of leakage rather than a real result.

So below I only use information that would be known before a game sells a single copy: platform, genre, publisher, and release year.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

model_df = clean_df.dropna(subset=['Publisher']).copy()

# Bucket long-tail publishers so the encoded feature space stays manageable
top_publishers = model_df['Publisher'].value_counts().head(20).index
model_df['Publisher_Bucketed'] = model_df['Publisher'].where(model_df['Publisher'].isin(top_publishers), 'Other')

features = model_df[['Platform', 'Genre', 'Year', 'Publisher_Bucketed']]
X = pd.get_dummies(features, columns=['Platform', 'Genre', 'Publisher_Bucketed'], drop_first=True)
y = model_df['Global_Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(f"R-squared: {r2_score(y_test, predictions):.3f}")
print(f"MAE:       {mean_absolute_error(y_test, predictions):.3f}M units")
print(f"RMSE:      {mean_squared_error(y_test, predictions) ** 0.5:.3f}M units")

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(9, 5))
importances.iloc[::-1].plot(kind='barh', color='teal')
plt.title('Top Factors Influencing Global Sales\n(Platform / Genre / Publisher / Year only, no regional sales leakage)', fontweight='bold')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()


With the leakage removed, the model only explains about 5% of the variance (R² ≈ 0.05, MAE ≈ 0.5M units). That's a pretty weak model, but I think it's an interesting result. It suggests platform, genre, publisher, and release year alone don't tell you much about whether a game becomes a hit. Year and being published by Nintendo carry the most signal, but not by much. Commercial success likely depends a lot more on things this dataset doesn't have, like review scores, marketing, or being part of an established franchise.

## 7. A Quick Time-Series Forecast (1996-2015)

The Random Forest in Section 6 treats `Year` as just another feature. It doesn't actually model the fact that this year's sales probably depend on last year's, the way an actual time series would. As a follow-up, I fit a simple exponential smoothing model on the 1996-2015 window (20 annual totals) and forecast a few years out, to see what treating this properly as a time series shows.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from statsmodels.tsa.holtwinters import ExponentialSmoothing

annual_sales = clean_df[(clean_df['Year'] >= 1996) & (clean_df['Year'] <= 2015)].groupby('Year')['Global_Sales'].sum()
annual_sales.index = pd.date_range(start='1996', periods=len(annual_sales), freq='YS')

ts_model = ExponentialSmoothing(annual_sales, trend='add', damped_trend=True).fit()
ts_forecast = ts_model.forecast(5)

plt.figure(figsize=(11, 5))
plt.plot(annual_sales.index.year, annual_sales.values, marker='o', label='Actual (1996-2015)')
plt.plot(ts_forecast.index.year, ts_forecast.values, marker='o', linestyle='--', color='firebrick', label='Forecast (2016-2020)')
plt.title('Global Sales: Damped Holt-Winters Forecast', fontweight='bold')
plt.ylabel('Global Sales (Millions)')
plt.xlabel('Year')
plt.legend()
plt.tight_layout()
plt.show()

print(ts_forecast.round(1))


The model projects a continued decline through 2020, which lines up well with how thin the data gets after 2015 (see Section 10) rather than necessarily reflecting a real decline in the games market.

## 8. Clustering Publishers

Section 3 split publishers into hit-makers and volume-dealers just by looking at the scatter plot. Here I run k-means on `Game_Count` and `Sales_Per_Game` (standardized) for the same set of publishers (≥10 titles) to see if a similar grouping shows up automatically.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

cluster_pubs = publisher_stats[publisher_stats['Game_Count'] >= 10].copy()
X_scaled = StandardScaler().fit_transform(cluster_pubs[['Game_Count', 'Sales_Per_Game']])

# quick elbow check to justify the number of clusters
inertias = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled).inertia_ for k in range(2, 7)]
plt.figure(figsize=(6, 3.5))
plt.plot(range(2, 7), inertias, marker='o')
plt.title('Elbow Check')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.tight_layout()
plt.show()

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_scaled)
cluster_pubs['Cluster'] = kmeans.labels_

plt.figure(figsize=(10, 6))
sns.scatterplot(data=cluster_pubs, x='Game_Count', y='Sales_Per_Game', hue='Cluster', palette='deep', s=70)
for name, row in cluster_pubs.sort_values('Total_Sales', ascending=False).head(10).iterrows():
    plt.annotate(name, (row['Game_Count'], row['Sales_Per_Game']), xytext=(6, 4), textcoords='offset points', fontsize=8)
plt.title('Publisher Clusters (k-means, k=3)', fontweight='bold')
plt.xlabel('Number of Titles Released')
plt.ylabel('Average Global Sales per Title (Millions)')
plt.tight_layout()
plt.show()

print(cluster_pubs.groupby('Cluster')[['Game_Count', 'Sales_Per_Game', 'Total_Sales']].mean().round(2))
print(cluster_pubs.groupby('Cluster').size().rename('Publishers'))


Three clusters come out fairly cleanly, and the elbow chart doesn't strongly argue for more than that. There's a small group (Nintendo, Take-Two, Microsoft Game Studios, Square Enix) with mid-sized catalogs and high sales per game. Then a large group of mostly smaller/niche publishers with low output on both counts. And a separate group (EA, Activision, Sony, Ubisoft) with huge catalogs but more moderate averages. That's basically the same story as the manual scatter-plot read from Section 3, which is reassuring since it holds up even when the clustering finds the groups on its own instead of me picking what looked like an outlier.

## 9. Does Critic Score Help Predict Sales?

The Future Work list above called out review scores as the most likely missing variable in Section 6's model, so I went and found some. I'm pulling in critic and user scores from a companion dataset (games originally scraped from Metacritic) and joining it in by Name and Platform. It only covers about half of `clean_df`'s rows, since plenty of these games never got a Metacritic page, so this section runs on a smaller, review-heavier sample than the rest of the notebook.

In [ ]:
ratings = pd.read_csv('https://raw.githubusercontent.com/Bakikhan/Video-Game-Sales-Dataset/main/Video_Games.csv')
ratings['User_Score'] = pd.to_numeric(ratings['User_Score'], errors='coerce')  # a handful of rows just say 'tbd'
ratings = ratings.drop_duplicates(subset=['Name', 'Platform'], keep='first')

scored_df = clean_df.merge(ratings[['Name', 'Platform', 'Critic_Score', 'User_Score']], on=['Name', 'Platform'], how='left')
coverage = scored_df['Critic_Score'].notna().mean()
print(f"{coverage:.1%} of clean_df rows matched to a critic score")


### Re-running the model with Critic_Score added

To make this a fair comparison I'm re-fitting the Section 6 model twice on the exact same (smaller) subset of games that actually has a critic score, once without `Critic_Score` as a feature and once with it. Comparing straight to the original R² ≈ 0.05 wouldn't really be fair since that used a bigger, different set of rows.

In [ ]:
score_model_df = scored_df.dropna(subset=['Publisher', 'Critic_Score']).copy()
top_publishers_scored = score_model_df['Publisher'].value_counts().head(20).index
score_model_df['Publisher_Bucketed'] = score_model_df['Publisher'].where(
    score_model_df['Publisher'].isin(top_publishers_scored), 'Other')

y_scored = score_model_df['Global_Sales']

# Same features as Section 6, just on this smaller subset
base_features = score_model_df[['Platform', 'Genre', 'Year', 'Publisher_Bucketed']]
X_base = pd.get_dummies(base_features, columns=['Platform', 'Genre', 'Publisher_Bucketed'], drop_first=True)
Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_base, y_scored, test_size=0.2, random_state=42)
model_base = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1).fit(Xb_train, yb_train)
r2_base = r2_score(yb_test, model_base.predict(Xb_test))

# Same, plus Critic_Score
full_features = score_model_df[['Platform', 'Genre', 'Year', 'Publisher_Bucketed', 'Critic_Score']]
X_full = pd.get_dummies(full_features, columns=['Platform', 'Genre', 'Publisher_Bucketed'], drop_first=True)
Xf_train, Xf_test, yf_train, yf_test = train_test_split(X_full, y_scored, test_size=0.2, random_state=42)
model_full = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1).fit(Xf_train, yf_train)
r2_full = r2_score(yf_test, model_full.predict(Xf_test))

print(f"R-squared without Critic_Score: {r2_base:.3f}")
print(f"R-squared with Critic_Score:    {r2_full:.3f}")

importances_full = pd.Series(model_full.feature_importances_, index=X_full.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(9, 5))
importances_full.iloc[::-1].plot(kind='barh', color='teal')
plt.title('Top Factors Influencing Global Sales, With Critic_Score Added', fontweight='bold')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()


R² roughly triples once Critic_Score is added, going from about 0.06 up to about 0.18, and it's clearly doing most of the work. It's the single most important feature by a wide margin, ahead of Year and Publisher. That's still a weak model in an absolute sense (plenty of what makes a game sell obviously isn't in either dataset), but it's a real jump from the baseline in Section 6.

 It's at least as likely that Critic_Score and Global_Sales are both downstream of the same thing, like how good the game actually was, how much marketing push it got, or whether it was a sequel to something people already loved. Critic_Score is probably better thought of as a stand-in for overall game quality, something platform/genre/publisher/year could never capture on their own.

## 10. Discussion

Putting the sections together:

Action and Sports are the largest genres by far, and that dominance has done nothing but grown over time. The market doesn't seem to be diversifying. Regional differences are real at the genre level even though they mostly wash out in the aggregate split (Japan and Role-Playing games, for instance). Publishers seem to fall into two pretty different strategies: Nintendo's smaller catalog of higher-performing titles versus EA/Activision's much larger catalog of more average-performing ones. It's not obvious a middle-ground approach is actually better than committing to one. Probably the most interesting finding is the modeling section: platform, genre, publisher, and year explain very little of why a specific game sells well, which lines up with the industry generally being described as "hit-driven." The time-series forecast (Section 7) and publisher clustering (Section 8) mostly confirm what the earlier sections already suggested rather than turning up anything new, which I think is a reasonable outcome for a second pass at the same questions with slightly more rigorous methods. Critic_Score (Section 9) is the one piece that actually moved the needle on the modeling question, tripling R² versus the metadata-only version, which suggests "how good the game is" matters a lot more than "what platform/genre/publisher/year it is," even if I can't say the relationship is causal.

## 11. Limitations

A few things worth keeping in mind here:

- The dataset gets noticeably sparser after 2015. Sales drop from 264M (2015) to 71M (2016) to almost nothing after that, which is very likely a data collection cutoff rather than an actual market collapse. I tried to keep the analysis focused on the well-covered 1996-2015 window for anything year-related.
- The 271 release years I filled in via the RAWG API came from fuzzy title matching, so a handful are probably matched to the wrong edition or region of a game.
- Global_Sales is a derived total, not an independent number. That's why I excluded regional sales as model features (Section 6).
- VGChartz numbers are themselves estimates, not verified retail data, so I'd treat any specific number here as directional rather than exact.
- The critic-score match in Section 9 only covers about half of `clean_df`, and it's not a random half: older, smaller, and more obscure titles are less likely to have a Metacritic page in the first place, so that section's model is trained on a review-heavier slice of the catalog than the rest of the notebook.

## 12. Export for Tableau

Reshaping the cleaned data into long format (one row per title/region) in case I want to explore it further in Tableau.

In [ ]:
core_cols = ['Name', 'Platform', 'Year', 'Genre', 'Publisher']
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']

tableau_df = pd.melt(clean_df, id_vars=core_cols, value_vars=sales_cols,
                      var_name='Region', value_name='Sales')
tableau_df['Region'] = tableau_df['Region'].str.replace('_Sales', '', regex=False)
tableau_df = tableau_df[tableau_df['Sales'] > 0]

tableau_df.to_csv('vgsales_for_tableau.csv', index=False)
print(f"Exported {len(tableau_df):,} rows to 'vgsales_for_tableau.csv'")
